In [2]:
import gurobipy as gp
import gurobi_utils as gu
import miplib_loader as ml
import example_loader as el
import numpy as np
import scipy.sparse as sps
import scipy.sparse.linalg as spsl

In [9]:
# A function to create cuts given a target point
def add_balas_ball_cut(relaxed: gp.Model, target, integer_vars, integer_idx):
    # for each column in the tableau
    # construct a sparse vector for it
    # get the length of that vector via norm1 (plus 1 if we're an int column)
    # add our cut: sum_j(x_j/a_j)
    
    norm = 2
    current = integer_vars.X
    radius = np.linalg.norm(current - target, norm)
    if radius <= relaxed.params.FeasibilityTol:
        return False  # TODO: tolerance should apply to each component individually?
    
    print("   Gap to target:", radius, ":", current[:7], "to", target[:7])
    
    basis = gu.read_basis(relaxed)
    tableau, col_to_var = gu.read_tableau(relaxed, basis, extra_rows=1)
    
    # drop the rows of non-integer variables:
    to_drop = [i for i, base in enumerate(basis) if base not in integer_idx]
    tableau = np.delete(tableau, to_drop, axis=0)

    # Conforti has negative vectors with 1 at row=col, with the rest negated:
    tableau = -tableau
    int_cols = [i for i, c in enumerate(col_to_var) if c in integer_idx]
    tableau[-1, int_cols] = 1  # use our extra row to store the col==row -> 1
  
    lengths = np.linalg.norm(tableau, norm, axis=0)
        
    variables = relaxed.getVars()
    constraints = relaxed.getConstrs()
    def find_variable(index):
        if index < len(variables):
            return variables[index]
        # if only gurobi gave us access to their slack variables...
        # instead, we have to solve for it:
        constraint = constraints[index - len(variables)]
        if constraint.Sense != '>':
            assert constraint.Sense == '='
            return 0.0  # ignore equality constraints with slacks (as gurobi generates a slack for every constraint)
        lhs, rhs = relaxed.getRow(constraint), constraint.RHS
        return lhs - rhs
    
    relaxed.addConstr(gp.quicksum(lengths[i] * find_variable(j) for i, j in enumerate(col_to_var)) >= 1)
    return True

# a function to run cuts against the nearest integer:
def run_cuts_to_nearest_int(instances, cut_function, loops=8):
    for instance in instances:
        m: gp.Model = instance.as_gurobi_model()
        print("Running model:", m.ModelName)
        m.params.LogToConsole = 0
        gu.standardize_lt_to_gt(m)
        int_vars, int_idx = gu.relax_int_or_bin_to_continuous(m)
        for i in range(loops):
            m.optimize()
            nearest = gu.nearest_integer(int_vars)
            if not cut_function(m, nearest, int_vars, int_idx):
                print("Final score of relaxed:", m.getObjective().getValue())
                break

# test the cuts on simple examples:
run_cuts_to_nearest_int(el.get_instances().values(), add_balas_ball_cut)

Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Running model: 2D from bottom (manual slacks)
   Negated 0 constraints on 2D from bottom (manual slacks)
   Relaxed 2 variables on 2D from bottom (manual slacks)
   Gap to target: 0.4006168083848878 : [1.22222222 2.33333333] to [1. 2.]
   Gap to target: 0.6122251430986051 : [0.51511544 1.62622655] to [1. 2.]
   Gap to target: 0.680392542898281 : [1.51024877 1.52768893] to [2. 2.]
   Gap to target: 0.3728716579244233 : [1.87654938 1.3518426 ] to [2. 1.]
   Gap to target: 0.2574271224178971 : [0.8840219  1.22982124] to [1. 1.]
   Gap to target: 0.20996941790396909 : [0.20129621 1.0597243 ] to [-

In [8]:
# a function to find the hyperplane closest to a point
def compute_hyperplane_via_lp(x0, b0, tableau, basis, col_to_var, int_idx):
    m = gp.Model()
    b = m.addVar(lb=-gp.GRB.INFINITY)
    w = m.addMVar(x0.shape, lb=-gp.GRB.INFINITY)
    wn = m.addVar()
    m.addConstr(wn == gp.norm(w, 1))
    m.addConstr(wn == 1)  # optional
    z = m.addVar()
    m.addConstr(z >= x0 @ w - b - b0)
    m.addConstr(z >= -x0 @ w + b + b0)
    m.setObjective(z)
    # our w represents all integer variables,
    # so not all columns in the tableau have a corresponding integer var.
    for i, vec in enumerate(tableau.T):
        cv = col_to_var[i]
        w_idx = int_idx.get(cv, -1)
        if w_idx >= 0:
            m.addConstr(vec[:-1] @ w[basis] + vec[-1] * w[w_idx] <= b)
        else:
            m.addConstr(vec[:-1] @ w[basis] <= b)
    m.params.LogToConsole = 0
    m.optimize()
    assert m.Status == gp.GRB.OPTIMAL
    return w.X, b.X

# create a function to do cuts via LP:
def add_lp_ball_cut(relaxed: gp.Model, target, integer_vars, integer_idx):
    
    norm = 1
    current = integer_vars.X
    radius = np.linalg.norm(current - target, norm)
    if radius <= relaxed.params.FeasibilityTol:
        return False  # TODO: tolerance should apply to each component individually?
    
    print("   Gap to target:", radius, ":", current[:7], "to", target[:7])
    
    basis = gu.read_basis(relaxed)
    tableau, col_to_var = gu.read_tableau(relaxed, basis, extra_rows=1)
    
    # drop the rows of non-integer variables:
    to_drop = [i for i, base in enumerate(basis) if base not in integer_idx]
    tableau = np.delete(tableau, to_drop, axis=0)
    basis = np.delete(basis, to_drop)
    
    # integer_idx goes from var num to index in integer_vars
    int_cols = [i for i, c in enumerate(col_to_var) if c in integer_idx]
    tableau[-1, int_cols] = -1  # use our extra row to store the col==row -> 1
    lengths = np.linalg.norm(tableau, norm, axis=0)
    
    # normalize the columns:
    tableau *= radius
    tableau /= lengths

    # left off: tableau is the wrong size; needs to be full vector or lp needs to be smarter

    # generate the LP to find the hyperplane
    w, b = compute_hyperplane_via_lp(target - current, radius, tableau, basis, col_to_var, integer_idx)

    # add the cut:
    relaxed.addConstr(w @ (integer_vars - current) >= b)
    return True

run_cuts_to_nearest_int(el.get_instances().values(), add_lp_ball_cut)

Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Set parameter Presolve to value 0
Set parameter Heuristics to value 0
Set parameter Cuts to value 0
Running model: 2D from bottom (manual slacks)
   Negated 0 constraints on 2D from bottom (manual slacks)
   Relaxed 2 variables on 2D from bottom (manual slacks)
   Gap to target: 0.5555555555555556 : [1.22222222 2.33333333] to [1. 2.]
   Gap to target: 0.5555555555555558 : [1.22222222 2.33333333] to [1. 2.]
   Gap to target: 0.5555555555555558 : [1.22222222 2.33333333] to [1. 2.]
   Gap to target: 0.5555555555555558 : [1.22222222 2.33333333] to [1. 2.]
   Gap to target: 0.5555555555555558 : [1.22222222 2.33333333] to [1. 2.]
   Gap to target: 0.5555555555555558 : [1.22222222 2.33333333] to [1